# Parker Aerospace Value Deck Generator

Builds the customer deck from live governed tables. Every number is queried at run time.

* **Story:** Parker Aerospace is a component OEM, not an airline. It carries warranty, AOG-support and reputational cost for components that operators and MROs maintain. The deck is anchored on that position.
* **Metric owners** are named by role on every KPI.
* **Value section is written twice:** once in financial terms for the funding executive, once in operational terms for the maintenance / product-support leader.
* **Before/after numbers** come from the back-test in `Parker Component Failure Forecast` (`ml_component_failure_forecast`), with assumptions stated on the slide.

Output: `/Workspace/Users/ayman.el-ghazali@databricks.com/FE Bar/Parker_Aerospace_Value_Deck.pptx`

In [0]:
%pip install -q python-pptx
dbutils.library.restartPython()

In [0]:
C   = "genie_zeroops_mfg_catalog.default"
CUT = "2026-06-01"
INTERVENTION_SUCCESS = 0.60     # share of caught events a proactive visit converts to a planned repair (assumption)
RUN_COST_PER_YEAR    = 250_000  # platform + part-time analyst (assumption, for payback only)

q = lambda s: spark.sql(s).first().asDict()
base = q(f"""SELECT count(*) components, count(DISTINCT airline) operators, count(DISTINCT tail_number) aircraft,
                    round(sum(estimated_warranty_exposure_usd)) exposure_usd, count_if(failure_risk_category='CRITICAL') critical
             FROM {C}.gold_component_health""")
recs = q(f"""SELECT count(*) records, avg(cast(anomaly_flag AS INT)) anomaly_rate,
                    avg(CASE WHEN anomaly_flag THEN days_undetected END) days_undetected,
                    avg(cast(is_overdue AS INT)) overdue_rate
             FROM {C}.bronze_maintenance_records""")
fc = spark.table(f"{C}.ml_component_failure_forecast").toPandas()
ev = spark.sql(f"""SELECT m.component_id, r.unit_cost_usd, m.days_undetected, m.flight_delay_risk
                    FROM {C}.bronze_maintenance_records m JOIN {C}.bronze_components_registry r USING (component_id)
                    WHERE m.maintenance_date >= '{CUT}'
                      AND ((m.anomaly_flag AND m.anomaly_severity IN ('HIGH','CRITICAL')) OR m.warranty_claim_risk)""").toPandas()
flagged = set(fc.loc[fc.risk_decile == 10, "component_id"])
ev["caught"] = ev.component_id.isin(flagged)
aog = ev.flight_delay_risk.isin(["HIGH", "CRITICAL"])

k = dict(
    events=len(ev), caught=int(ev.caught.sum()), flagged=len(flagged),
    q_exposure=float(ev.unit_cost_usd.sum()), q_caught=float(ev.loc[ev.caught, "unit_cost_usd"].sum()),
    aog=int(aog.sum()), aog_caught=int((aog & ev.caught).sum()),
)
k["q_avoided"]   = k["q_caught"] * INTERVENTION_SUCCESS
k["y_avoided"]   = k["q_avoided"] * 4
k["q_after"]     = k["q_exposure"] - k["q_avoided"]
k["aog_after"]   = k["aog"] - k["aog_caught"] * INTERVENTION_SUCCESS
k["capture"]     = k["caught"] / k["events"]
k["payback_mo"]  = RUN_COST_PER_YEAR / k["y_avoided"] * 12
from sklearn.metrics import roc_auc_score
k["auc"] = roc_auc_score(fc.actual_next_quarter_event, fc.forecast_probability)

top_ops = spark.sql(f"""SELECT airline, round(avg(cast(anomaly_flag AS INT))*100,1) anomaly_pct, round(avg(cast(is_overdue AS INT))*100,1) overdue_pct
                        FROM {C}.bronze_maintenance_records GROUP BY 1 ORDER BY 2 DESC LIMIT 3""").toPandas()
print(base, recs, {a: (round(b, 3) if isinstance(b, float) else b) for a, b in k.items()}, sep="\n")
display(top_ops)

In [0]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

NAVY, ORANGE, WHITE, GREY, DARK = RGBColor(0x0B,0x1F,0x3A), RGBColor(0xF2,0x8C,0x28), RGBColor(0xFF,0xFF,0xFF), RGBColor(0xB8,0xC2,0xCC), RGBColor(0x22,0x22,0x22)
prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
BLANK = prs.slide_layouts[6]
usd = lambda v: f"${v/1e6:.2f}M" if v >= 1e6 else f"${v/1e3:,.0f}K"

def slide(title, subtitle=None):
    s = prs.slides.add_slide(BLANK)
    s.background.fill.solid(); s.background.fill.fore_color.rgb = NAVY
    tb = s.shapes.add_textbox(Inches(0.6), Inches(0.35), Inches(12), Inches(0.9)).text_frame
    tb.text = title; tb.paragraphs[0].runs[0].font.size = Pt(32); tb.paragraphs[0].runs[0].font.bold = True; tb.paragraphs[0].runs[0].font.color.rgb = WHITE
    if subtitle:
        p = tb.add_paragraph(); p.text = subtitle; p.runs[0].font.size = Pt(16); p.runs[0].font.color.rgb = ORANGE
    bar = s.shapes.add_shape(1, Inches(0), Inches(7.2), prs.slide_width, Inches(0.3)); bar.fill.solid(); bar.fill.fore_color.rgb = ORANGE; bar.line.fill.background()
    return s

def bullets(s, items, top=1.6, left=0.6, width=12, size=18, color=WHITE):
    tf = s.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(5.4)).text_frame; tf.word_wrap = True
    for i, it in enumerate(items):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = it; p.space_after = Pt(10)
        for r in p.runs: r.font.size = Pt(size); r.font.color.rgb = color

def table(s, rows, top=1.7, col_w=None, size=14, height=0.45):
    n, m = len(rows), len(rows[0]); col_w = col_w or [12.1 / m] * m
    t = s.shapes.add_table(n, m, Inches(0.6), Inches(top), Inches(sum(col_w)), Inches(height * n)).table
    for j, w in enumerate(col_w): t.columns[j].width = Inches(w)
    for i, row in enumerate(rows):
        for j, v in enumerate(row):
            c = t.cell(i, j); c.text = str(v)
            c.fill.solid(); c.fill.fore_color.rgb = ORANGE if i == 0 else (RGBColor(0xF4,0xF6,0xF8) if i % 2 else WHITE)
            for p in c.text_frame.paragraphs:
                for r in p.runs: r.font.size = Pt(size); r.font.bold = (i == 0); r.font.color.rgb = WHITE if i == 0 else DARK
    return t

def note(s, text, top=6.55):
    tf = s.shapes.add_textbox(Inches(0.6), Inches(top), Inches(12), Inches(0.6)).text_frame; tf.word_wrap = True
    tf.text = text; tf.paragraphs[0].runs[0].font.size = Pt(11); tf.paragraphs[0].runs[0].font.color.rgb = GREY

# 1 Title
s = slide("Parker Aerospace: Installed-Base Reliability & Warranty Intelligence",
          "Predict which Parker components will fail in service next quarter, and act before the operator calls an AOG")
bullets(s, ["Prepared for: VP Aftermarket & Customer Support (economic buyer) and Director, Product Support Engineering (operational owner)",
            "Built on Databricks: Lakeflow pipelines, Unity Catalog, MLflow, Vector Search, Genie, Lakebase, Databricks Apps",
            "All figures are from a synthetic installed base modelled on Parker's aerospace portfolio"], top=2.6, size=18)

# 2 Value-chain position
s = slide("Parker's position in the aerospace value chain", "Parker builds the part. The operator and its MRO maintain it. Parker pays when it fails early.")
table(s, [["Party", "Role", "What it controls", "What it costs Parker when it goes wrong"],
          ["Parker Aerospace (Tier-1 OEM)", "Designs/builds hydraulic actuators, fuel valves, pneumatic valves, ECS controllers, sensors", "Design, CMMs, service bulletins, warranty terms", "Warranty claims, AOG spares + field-service dispatch"],
          ["Airframer", "Integrates Parker systems into the aircraft", "Supplier scorecard, reliability guarantees", "Reliability-guarantee penalties, share of future platforms"],
          [f"{base['operators']} airline operators", f"Fly {base['aircraft']} aircraft carrying {base['components']:,} Parker components", "Flight hours, cycles, maintenance schedule", "Unscheduled removals reported against Parker part numbers"],
          ["MRO shops", "Perform scheduled service and shop visits", "Service quality and record accuracy", "Missed or mis-recorded service turns into early failures and disputed claims"]],
      col_w=[2.6, 3.6, 2.9, 3.0], size=12, height=0.8)
note(s, "The gap: Parker carries the warranty and AOG risk but sees operator service data late and in pieces. This platform puts operator service records and component telemetry in one governed view.")

# 3 Owners of the metrics
s = slide("Who owns these numbers", "Each KPI in this deck has a named executive owner (by role)")
table(s, [["KPI", "Owner (role)", "Why they care"],
          ["Warranty cost and exposure ($)", "VP Aftermarket & Customer Support", "Owns the aftermarket P&L and the warranty reserve"],
          ["Program funding, payback", "VP Finance, Aerospace Group", "Approves the investment and the reserve assumptions"],
          ["Unscheduled removals, anomaly rate, MTBUR", "Director, Product Support Engineering", "Owns reliability, fault isolation and CMM content"],
          ["AOG events, field-service dispatches", "Director, Customer Support & Field Service", "Owns response time and operator satisfaction"],
          ["Operator service compliance", "Director, Customer Support & Field Service", "Negotiates service obligations with each operator"]],
      col_w=[4.2, 3.9, 4.0], size=14, height=0.55)

# 4 Current state
s = slide("Today: reactive, late, and expensive", "Live from the governed gold tables")
table(s, [["Current-state metric", "Value", "Owner"],
          ["Installed Parker components monitored", f"{base['components']:,} on {base['aircraft']} aircraft, {base['operators']} operators", "VP Aftermarket"],
          ["Service records analysed (18 months)", f"{recs['records']:,}", "Dir. Product Support Eng."],
          ["Service events with a maintenance anomaly", f"{recs['anomaly_rate']:.1%}", "Dir. Product Support Eng."],
          ["Average days an anomaly goes undetected", f"{recs['days_undetected']:.0f} days", "Dir. Customer Support & Field Service"],
          ["Modelled warranty exposure on at-risk components", usd(base['exposure_usd']), "VP Aftermarket"],
          ["HIGH/CRITICAL service events or warranty-risk events per quarter", f"{k['events']} events, {usd(k['q_exposure'])} unit value", "VP Aftermarket"]],
      col_w=[5.6, 3.8, 2.7], size=14, height=0.55)
note(s, f"Worst operators by anomaly rate: " + ", ".join(f"{r.airline} {r.anomaly_pct}%" for r in top_ops.itertuples()))

# 5 Solution
s = slide("What we built", "One governed platform from operator data to an engineer's next action")
bullets(s, ["Ingest: Lakeflow pipelines land operator flight telemetry and Parker service records (bronze -> silver -> gold) with data-quality expectations",
            "Govern: Unity Catalog comments, quality tags, owner properties, operator-level row filters and a warranty-$ column mask, so an airline only sees its own fleet",
            f"Forecast: next-quarter failure model, point-in-time features, 5-fold CV, tracked in MLflow; ranks all {base['components']:,} components",
            "Act: Databricks App with dashboard, Genie (ask in plain English), and Maintenance Manual search (Vector Search) that returns the fault-isolation steps, part numbers and torque values",
            "Serve: Lakebase for low-latency operational reads from the app"], size=18)

# 6 Model honesty
s = slide("How good is the forecast? (back-test, Jun-Aug 2026)", "Trained only on data before 1 Jun 2026, scored against what actually happened next")
table(s, [["Approach", "Inspect 10% of fleet and catch...", "Lift vs random"],
          ["Random inspection", "10% of next-quarter events", "1.0x"],
          ["Rule: components with prior HIGH/CRITICAL history", "12% of events", "1.2x"],
          ["Databricks forecast (top decile)", f"{k['capture']:.0%} of events ({k['caught']} of {k['events']})", f"{k['capture']/0.10:.1f}x"]],
      col_w=[5.2, 4.6, 2.3], size=15, height=0.6)
note(s, f"ROC-AUC {k['auc']:.2f}. An earlier model reported 0.999 AUC; that was target leakage (the label was a threshold on one of its own inputs) and has been replaced. "
        "The signal is limited by the synthetic data. Real operator data is expected to carry stronger wear and overdue signals.", top=5.0)

# 7 Before / after
s = slide("Before and after: one quarter, back-tested", "Proactive field-service visits on the top-decile list")
table(s, [["KPI (owner)", "Before (today)", "After", "Change"],
          ["Quarterly exposure from HIGH/CRITICAL + warranty-risk events (VP Aftermarket)", usd(k['q_exposure']), usd(k['q_after']), f"-{k['q_avoided']/k['q_exposure']:.0%}"],
          ["Events found before they happen (Dir. Product Support Eng.)", "0", f"{k['caught']} of {k['events']}", f"+{k['capture']:.0%} caught"],
          ["Events carrying HIGH/CRITICAL AOG risk (Dir. Customer Support)", f"{k['aog']}", f"{k['aog_after']:.0f}", f"-{(k['aog']-k['aog_after'])/k['aog']:.0%}"],
          ["Days undetected for flagged components (Dir. Customer Support)", f"{recs['days_undetected']:.0f} days", "0 days (found at visit)", f"-{recs['days_undetected']:.0f} days"],
          ["Components Field Service must visit (Dir. Customer Support)", "unknown / blanket checks", f"{k['flagged']} per quarter", "targeted list"]],
      col_w=[5.4, 2.3, 2.3, 2.1], size=14, height=0.6)
note(s, f"Assumption: {INTERVENTION_SUCCESS:.0%} of caught events are converted to a planned repair by the proactive visit. Exposure is unit replacement value of the affected component.")

# 8 Funding executive pass
s = slide("For the VP Finance / VP Aftermarket: the financial case", "Warranty reserve, cash and payback")
bullets(s, [f"Avoided warranty exposure: {usd(k['q_avoided'])} per quarter, about {usd(k['y_avoided'])} per year, on today's {base['components']:,}-component synthetic installed base",
            f"Quarterly exposure from high-severity events moves from {usd(k['q_exposure'])} to {usd(k['q_after'])} (-{k['q_avoided']/k['q_exposure']:.0%})",
            f"Payback: about {k['payback_mo']:.0f} months at an assumed {usd(RUN_COST_PER_YEAR)}/year run cost (platform + part-time analyst)",
            "Reserve accuracy: per-component expected exposure (probability x unit cost) replaces a flat historical accrual, so the warranty reserve can be set bottom-up",
            "Scales linearly: the same pipeline and model cover every additional operator contract without new headcount",
            "Ask: fund a 90-day pilot with the two highest-anomaly operators, with the success gate defined on the next slide"], size=18)

# 9 Maintenance leader pass
s = slide("For the Director, Product Support Engineering: the operational case", "Fewer surprises, faster fault isolation, a short list to act on")
bullets(s, [f"A ranked list each quarter: {k['flagged']} components to visit instead of blanket checks, and that list holds {k['capture']:.0%} of next quarter's high-severity events",
            f"AOG-risk events drop from {k['aog']} to about {k['aog_after']:.0f} per quarter. Each one avoided is one fewer emergency spares shipment and field-service dispatch",
            f"Anomalies on flagged components are found at the visit, not about {recs['days_undetected']:.0f} days later at failure",
            "Engineers search the Maintenance Manuals by symptom or part number and get the fault-isolation steps, limits and torque values in seconds",
            "Ask Genie in plain English: 'Which SkyWest hydraulic actuators are overdue?' Answers come from the same governed tables",
            "Operators can be given read access to their own fleet only (row filters), so findings can be shared without exposing other customers"], size=18)

# 10 Pilot
s = slide("Next step: 90-day pilot", "Owner: VP Aftermarket & Customer Support. Sponsor: VP Finance, Aerospace Group")
table(s, [["Week", "Milestone", "Success gate"],
          ["0-4", f"Connect real service records for {top_ops.airline.iloc[0]} and {top_ops.airline.iloc[1]}", "Data lands daily and passes quality checks"],
          ["4-8", "Retrain on real data and back-test on the last quarter", "Top-decile list catches at least 2x the events random inspection does"],
          ["8-12", "Field Service works the list, engineers use manual search", "At least 25% fewer high-severity events on visited components"],
          ["12", "Decision review with VP Finance", "Annualised avoided exposure exceeds 3x run cost"]],
      col_w=[1.4, 6.0, 4.7], size=14, height=0.6)

OUT = "/Workspace/Users/ayman.el-ghazali@databricks.com/FE Bar/Parker_Aerospace_Value_Deck.pptx"
prs.save(OUT)
print(f"Saved {len(prs.slides)} slides -> {OUT}")